In [1]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd().resolve())

Current working directory:
/Users/arvin/Documents/dev_proj/Multi_Material_Neutron_Diffraction/notebooks


In [2]:
from pathlib import Path

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (
        (candidate / "README.md").is_file()
        and
        (candidate / "notebooks").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root.")

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "multi_material_diffraction_dataset"
)

print("Project root:")
print(PROJECT_ROOT)

print()

print("Dataset will be saved under:")
print(DATASET_ROOT)

Project root:
/Users/arvin/Documents/dev_proj/Multi_Material_Neutron_Diffraction

Dataset will be saved under:
/Users/arvin/Documents/dev_proj/Multi_Material_Neutron_Diffraction/data/multi_material_diffraction_dataset


In [ ]:
from pathlib import Path
from decimal import Decimal
import subprocess
import shutil
import re
import json
import hashlib


# =====================================================================
# =====================================================================
#
#   MULTI-MATERIAL NEUTRON POWDER DIFFRACTION DATASET
#
#   MATERIAL × LAMBDA × DLAMBDA
#
#   Outputs:
#
#       1. Original 2D diffraction patterns
#       2. Dynamic-range corrected PNG previews
#       3. 1D radial powder diffractograms I(2theta)
#       4. CSV + NPZ 1D profiles
#       5. Complete metadata
#
#   DIRECT BEAM IS PHYSICALLY REMOVED BY A BEAMSTOP
#
# =====================================================================
# =====================================================================


# =====================================================================
# 1. HOW MANY MATERIALS?
# =====================================================================
#
# Start with:
#
#       10
#
# Later just change:
#
#       number_of_materials = 20
#
# and the code will select the first 20 available materials.
#
# =====================================================================

number_of_materials = 10


# =====================================================================
# 2. WAVELENGTH SWEEP
# =====================================================================

lambda1 = 1.00          # Å
lambda2 = 2.00          # Å
lambda_step = 0.10      # Å


# =====================================================================
# 3. DLAMBDA SWEEP
# =====================================================================
#
# IMPORTANT:
#
# For Source_simple with gauss=0:
#
# lambda range for ONE simulation =
#
#       lambda0 - dlambda  -->  lambda0 + dlambda




# =====================================================================
# 3. DLAMBDA SWEEP
# =====================================================================
#
# IMPORTANT:
#
# dlambda is NOT the spacing between simulations.
#
# For Source_simple with gauss=0:
#
# wavelength range in each simulation =
#
#       lambda0 - dlambda
#              to
#       lambda0 + dlambda
#
#
# Example:
#
# lambda0 = 2.0 Å
# dlambda = 0.05 Å
#
# means source wavelengths:
#
#       1.95 Å ---> 2.05 Å
#
# ---------------------------------------------------------------------
#
# =====================================================================

dlambda1 = 0.01         # Å
dlambda2 = 0.1         # Å
dlambda_step = 0.01     # Å


# =====================================================================
# 4. MONTE CARLO SETTINGS
# =====================================================================
#
# Start reasonably small.
#
# If the diffraction rings are statistically sparse later,
# increase ncount to 1_000_000.
#
# SPLIT improves statistics at the sample.
#
# =====================================================================

ncount = 1_000_000

sample_split = 20


# =====================================================================
# 5. SIMPLE DIFFRACTOMETER GEOMETRY
# =====================================================================


# ---------------------------------------------------------------------
# SOURCE
# ---------------------------------------------------------------------

source_radius = 0.020           # m = 20 mm


# ---------------------------------------------------------------------
# GUIDE
# ---------------------------------------------------------------------

source_to_guide = 0.50          # m

guide_length = 4.00             # m

guide_width = 0.040             # m = 40 mm
guide_height = 0.040            # m = 40 mm

guide_m = 2.0


# ---------------------------------------------------------------------
# COLLIMATION
# ---------------------------------------------------------------------

slit1_width = 0.010             # 10 mm
slit1_height = 0.010

slit2_width = 0.008             # 8 mm
slit2_height = 0.008


# ---------------------------------------------------------------------
# SAMPLE
# ---------------------------------------------------------------------

sample_radius = 0.005           # 5 mm radius
sample_height = 0.020           # 20 mm


# ---------------------------------------------------------------------
# DETECTOR
# ---------------------------------------------------------------------

sample_detector_distance = 0.50     # m

detector_width = 1.00               # m
detector_height = 1.00              # m

detector_nx = 512
detector_ny = 512


# ---------------------------------------------------------------------
# BEAMSTOP
# ---------------------------------------------------------------------
#
# VERY IMPORTANT:
#
# Circular beamstop.
#
# Diameter = 30 mm
# Radius   = 15 mm
#
# It is placed ONLY 20 mm before the detector.
#
# This blocks the direct beam without throwing away a huge
# low-angle diffraction cone.
#
# ---------------------------------------------------------------------

beamstop_radius = 0.015              # m = 15 mm radius

beamstop_detector_gap = 0.020        # m = 20 mm before detector


# =====================================================================
# 6. 1D DIFFRACTOGRAM SETTINGS
# =====================================================================
#
# Number of angular bins for radial integration.
#
# =====================================================================

profile_bins = 600


# =====================================================================
# 7. PROJECT AND DATASET LOCATION
# =====================================================================
#
# The notebook may be launched from:
#
#   project root/
#
# or:
#
#   project root/notebooks/
#
# We therefore locate the repository root explicitly instead of
# relying on the current working directory.
#
# =====================================================================


def find_project_root():

    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents
    ]:

        if (
            (candidate / "README.md").is_file()
            and
            (candidate / "notebooks").is_dir()
        ):

            return candidate


    raise RuntimeError(

        "\nCould not locate the project root.\n\n"
        "Expected to find a directory containing:\n"
        "    README.md\n"
        "    notebooks/\n"
    )


PROJECT_ROOT = find_project_root()


DATA_DIR = (
    PROJECT_ROOT
    /
    "data"
)


dataset_root = (
    DATA_DIR
    /
    "multi_material_diffraction_dataset"
)


instrument_file = (
    dataset_root
    /
    "simple_ncrystal_powder_diffractometer.instr"
)


monitor_filename = "rings.dat"


# ---------------------------------------------------------------------
# Create dataset root if necessary
# ---------------------------------------------------------------------

dataset_root.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------------------
# Existing-data behaviour
#
# False:
#     reuse an already simulated pattern when it belongs to the exact
#     same configuration.
#
# True:
#     simulate it again.
# ---------------------------------------------------------------------

overwrite_existing = False


# ---------------------------------------------------------------------
# Preview output
# ---------------------------------------------------------------------

save_2d_previews = True

save_1d_previews = True


# ---------------------------------------------------------------------
# Report project paths
# ---------------------------------------------------------------------

print("=" * 80)
print("PROJECT PATHS")
print("=" * 80)

print()

print("Current working directory:")
print(Path.cwd().resolve())

print()

print("Project root:")
print(PROJECT_ROOT)

print()

print("Dataset root:")
print(dataset_root)

print()

print("Generated McStas instrument:")
print(instrument_file)

print()

# =====================================================================
# 8. SOFTWARE CHECK
# =====================================================================

mcrun_executable = shutil.which(
    "mcrun"
)

nctool_executable = shutil.which(
    "nctool"
)


if mcrun_executable is None:

    raise RuntimeError(
        "\n'mcrun' was not found.\n\n"
        "Make sure the PaNRAID / McStas environment is active "
        "and that this Jupyter notebook is using that kernel."
    )


if nctool_executable is None:

    raise RuntimeError(
        "\n'nctool' was not found.\n\n"
        "NCrystal is not visible from this Jupyter environment."
    )


print("=" * 80)
print("SOFTWARE")
print("=" * 80)

print()

print("mcrun:")
print(mcrun_executable)

print()

print("nctool:")
print(nctool_executable)

print()


# =====================================================================
# 9. READ NCRYSTAL MATERIAL LIBRARY
# =====================================================================

browse_result = subprocess.run(

    [
        nctool_executable,
        "--browse"
    ],

    capture_output=True,

    text=True
)


if browse_result.returncode != 0:

    raise RuntimeError(
        "\nCould not browse NCrystal material library:\n\n"
        + browse_result.stderr
    )


browse_text = (

    browse_result.stdout
    +
    "\n"
    +
    browse_result.stderr
)


# ---------------------------------------------------------------------
# Extract every .ncmat filename reported by NCrystal
# ---------------------------------------------------------------------

all_ncmat_files = sorted(

    set(

        re.findall(

            r"[A-Za-z0-9_.+\-]+\.ncmat",

            browse_text
        )
    )
)


print(
    f"NCrystal reports {len(all_ncmat_files)} "
    f".ncmat files."
)


# =====================================================================
# 10. CURATED MATERIAL ORDER
# =====================================================================
#
# We define MATERIAL TYPES, not blindly fixed filenames.
#
# The regular expressions are matched against what YOUR installed
# NCrystal actually contains.
#
# First 10 are intended to be:
#
# Al
# Cu
# alpha-Fe
# Ni
# Si
# Ge
# Mg
# Zn
# Zr
# Cr
#
# More are included so later:
#
#       number_of_materials = 20
#
# can work automatically.
#
# =====================================================================

candidate_materials = [

    ("Al", [
        r"^Al_sg225.*\.ncmat$"
    ]),

    ("Cu", [
        r"^Cu_sg225.*\.ncmat$"
    ]),

    ("Fe_alpha", [
        r"^Fe_sg229.*\.ncmat$"
    ]),

    ("Ni", [
        r"^Ni_sg225.*\.ncmat$"
    ]),

    ("Si", [
        r"^Si_sg227.*\.ncmat$"
    ]),

    ("Ge", [
        r"^Ge_sg227.*\.ncmat$"
    ]),

    ("Mg", [
        r"^Mg_sg194.*\.ncmat$"
    ]),

    ("Zn", [
        r"^Zn_sg194.*\.ncmat$"
    ]),

    ("Zr", [
        r"^Zr_sg194.*\.ncmat$"
    ]),

    ("Cr", [
        r"^Cr_sg229.*\.ncmat$"
    ]),

    ("Ag", [
        r"^Ag_sg225.*\.ncmat$"
    ]),

    ("Au", [
        r"^Au_sg225.*\.ncmat$"
    ]),

    ("Pt", [
        r"^Pt_sg225.*\.ncmat$"
    ]),

    ("Pb", [
        r"^Pb_sg225.*\.ncmat$"
    ]),

    ("Mo", [
        r"^Mo_sg229.*\.ncmat$"
    ]),

    ("W", [
        r"^W_sg229.*\.ncmat$"
    ]),

    ("Ti", [
        r"^Ti_sg194.*\.ncmat$"
    ]),

    ("Al2O3", [
        r"^Al2O3_sg167.*\.ncmat$"
    ]),

    ("MgO", [
        r"^MgO_sg225.*\.ncmat$"
    ]),

    ("NaCl", [
        r"^NaCl_sg225.*\.ncmat$"
    ]),

    ("SiC_beta", [
        r"^SiC.*sg216.*\.ncmat$"
    ]),

    ("ZnO", [
        r"^ZnO_sg186.*\.ncmat$"
    ]),

    ("CaF2", [
        r"^CaF2_sg225.*\.ncmat$"
    ]),

    ("CeO2", [
        r"^CeO2_sg225.*\.ncmat$"
    ]),

    ("AlN", [
        r"^AlN_sg186.*\.ncmat$"
    ]),

    ("C_Diamond", [
        r"^C_sg227.*Diamond.*\.ncmat$",
        r"^C_sg227.*\.ncmat$"
    ]),

    ("C_Graphite", [
        r"^C_sg194.*[Gg]raphite.*\.ncmat$"
    ])
]


# =====================================================================
# 11. FIND THESE MATERIALS IN THE LOCAL NCRYSTAL LIBRARY
# =====================================================================

available_materials = []

used_files = set()


for label, patterns in candidate_materials:

    match = None

    for pattern in patterns:

        for filename in all_ncmat_files:

            if re.match(
                pattern,
                filename,
                flags=re.IGNORECASE
            ):

                match = filename

                break

        if match is not None:

            break


    if match is not None:

        if match not in used_files:

            available_materials.append(

                (
                    label,
                    match
                )
            )

            used_files.add(
                match
            )


# =====================================================================
# 12. IF NECESSARY, FILL WITH OTHER CRYSTALLINE NCRYSTAL MATERIALS
# =====================================================================
#
# This makes number_of_materials = 20, 25, etc. more robust.
#
# Only files containing "_sg" are considered here because these
# normally represent crystal structures with a space-group label.
#
# =====================================================================

if len(available_materials) < number_of_materials:

    for filename in all_ncmat_files:

        if filename in used_files:

            continue

        if "_sg" not in filename:

            continue

        label = Path(
            filename
        ).stem

        label = re.sub(
            r"[^A-Za-z0-9_]+",
            "_",
            label
        )

        available_materials.append(

            (
                label,
                filename
            )
        )

        used_files.add(
            filename
        )

        if len(available_materials) >= number_of_materials:

            break


if len(available_materials) < number_of_materials:

    raise RuntimeError(

        f"\nYou requested {number_of_materials} materials, "
        f"but only {len(available_materials)} suitable "
        f"NCrystal crystalline materials were found."
    )


materials = available_materials[
    :number_of_materials
]


print()

print("=" * 80)
print("MATERIALS THAT WILL ACTUALLY BE SIMULATED")
print("=" * 80)

print()


for i, (label, filename) in enumerate(
    materials,
    start=1
):

    print(
        f"{i:2d}. "
        f"{label:<20} -> {filename}"
    )


# =====================================================================
# 13. PARAMETER GRID
# =====================================================================

def make_grid(
    start,
    stop,
    step
):

    n_float = (

        (stop - start)
        /
        step
    )


    n = int(
        round(
            n_float
        )
    )


    if not np.isclose(
        n_float,
        n,
        rtol=1e-10,
        atol=1e-12
    ):

        raise ValueError(

            "\nUpper limit is not exactly reachable "
            "with the chosen step.\n\n"

            f"start = {start}\n"
            f"stop  = {stop}\n"
            f"step  = {step}"
        )


    return (

        start
        +
        np.arange(
            n + 1
        )
        *
        step
    )


lambdas = make_grid(

    lambda1,
    lambda2,
    lambda_step
)


dlambdas = make_grid(

    dlambda1,
    dlambda2,
    dlambda_step
)


# =====================================================================
# 14. SAFE NUMBER TAGS FOR FILENAMES
# =====================================================================

def decimals_from_step(
    step
):

    d = Decimal(
        str(step)
    ).normalize()

    return max(
        0,
        -d.as_tuple().exponent
    )


name_decimals = max(

    3,

    decimals_from_step(
        lambda_step
    ),

    decimals_from_step(
        dlambda_step
    )
)


def number_to_tag(
    value
):

    return (

        f"{value:.{name_decimals}f}"
        .replace(
            ".",
            "p"
        )
    )


# =====================================================================
# 15. DATASET SWEEP NAME
# =====================================================================
#
# The wavelength ranges alone are NOT sufficient to identify a dataset.
#
# If the geometry, materials, statistics, detector, beamstop, etc.
# change, the configuration ID also changes. This prevents an old
# simulation from being silently reused under a new configuration.
#
# =====================================================================


configuration = {

    # Materials
    "materials": materials,

    # Wavelength grid
    "lambda1_A": lambda1,
    "lambda2_A": lambda2,
    "lambda_step_A": lambda_step,

    "dlambda1_A": dlambda1,
    "dlambda2_A": dlambda2,
    "dlambda_step_A": dlambda_step,

    # Monte Carlo
    "ncount": ncount,
    "sample_split": sample_split,

    # Source
    "source_radius_m": source_radius,

    # Guide
    "source_to_guide_m": source_to_guide,
    "guide_length_m": guide_length,
    "guide_width_m": guide_width,
    "guide_height_m": guide_height,
    "guide_m": guide_m,

    # Collimation
    "slit1_width_m": slit1_width,
    "slit1_height_m": slit1_height,
    "slit2_width_m": slit2_width,
    "slit2_height_m": slit2_height,

    # Sample
    "sample_radius_m": sample_radius,
    "sample_height_m": sample_height,

    # Detector
    "sample_detector_distance_m": sample_detector_distance,
    "detector_width_m": detector_width,
    "detector_height_m": detector_height,
    "detector_nx": detector_nx,
    "detector_ny": detector_ny,

    # Beamstop
    "beamstop_radius_m": beamstop_radius,
    "beamstop_detector_gap_m": beamstop_detector_gap,

    # Radial integration
    "profile_bins": profile_bins
}


configuration_string = json.dumps(
    configuration,
    sort_keys=True
)


configuration_id = hashlib.sha256(
    configuration_string.encode("utf-8")
).hexdigest()[:10]


sweep_name = (

    f"lambda_"
    f"{number_to_tag(lambda1)}"
    f"_to_"
    f"{number_to_tag(lambda2)}"
    f"_step_"
    f"{number_to_tag(lambda_step)}"

    f"__dlambda_"
    f"{number_to_tag(dlambda1)}"
    f"_to_"
    f"{number_to_tag(dlambda2)}"
    f"_step_"
    f"{number_to_tag(dlambda_step)}"

    f"__cfg_"
    f"{configuration_id}"
)


# ---------------------------------------------------------------------
# Full path of this exact simulation sweep
# ---------------------------------------------------------------------

sweep_root = (

    dataset_root
    /
    sweep_name
)


sweep_root.mkdir(

    parents=True,
    exist_ok=True
)


print()

print("Sweep:")
print(sweep_name)

print()

print("Sweep directory:")
print(sweep_root.resolve())

print()
# =====================================================================
# 16. INSTRUMENT POSITIONS
# =====================================================================

guide_z = (
    source_to_guide
)


guide_exit_z = (
    guide_z
    +
    guide_length
)


slit1_z = (
    guide_exit_z
    +
    0.10
)


slit2_z = (
    guide_exit_z
    +
    0.40
)


sample_z = (
    guide_exit_z
    +
    0.50
)


detector_z = (

    sample_z
    +
    sample_detector_distance
)


# ---------------------------------------------------------------------
# Beamstop is CLOSE TO DETECTOR.
# ---------------------------------------------------------------------

beamstop_z = (

    detector_z
    -
    beamstop_detector_gap
)


beamstop_distance_from_sample = (

    beamstop_z
    -
    sample_z
)


# =====================================================================
# 17. BEAMSTOP ANGULAR CUTOFF
# =====================================================================
#
# Approximate minimum observable scattering angle:
#
# tan(2theta_min) = beamstop_radius / distance(sample -> beamstop)
#
# =====================================================================

beamstop_two_theta_deg = np.degrees(

    np.arctan2(

        beamstop_radius,

        beamstop_distance_from_sample
    )
)


# =====================================================================
# 18. MAXIMUM FULL-RING ANGLE
# =====================================================================
#
# We only radially integrate where an ENTIRE ring fits inside
# the square detector.
#
# =====================================================================

full_ring_radius = (

    min(
        detector_width,
        detector_height
    )
    /
    2
)


max_full_ring_two_theta_deg = np.degrees(

    np.arctan2(

        full_ring_radius,

        sample_detector_distance
    )
)


print()

print("=" * 80)
print("DIFFRACTOMETER GEOMETRY")
print("=" * 80)

print()

print(
    f"Guide:              "
    f"{guide_length:.2f} m long, "
    f"{guide_width*1000:.0f} × "
    f"{guide_height*1000:.0f} mm, "
    f"m={guide_m:g}"
)

print(
    f"Sample:             "
    f"{2*sample_radius*1000:.0f} mm diameter × "
    f"{sample_height*1000:.0f} mm high"
)

print(
    f"Detector:           "
    f"{detector_width:.2f} × "
    f"{detector_height:.2f} m, "
    f"{detector_nx} × {detector_ny} pixels"
)

print(
    f"Sample-detector:    "
    f"{sample_detector_distance:.3f} m"
)

print()

print(
    f"BEAMSTOP diameter:  "
    f"{2*beamstop_radius*1000:.1f} mm"
)

print(
    f"Beamstop-detector:  "
    f"{beamstop_detector_gap*1000:.1f} mm"
)

print(
    f"Beamstop cutoff:    "
    f"approximately "
    f"{beamstop_two_theta_deg:.2f}° in 2theta"
)

print()

print(
    f"Full-ring profile:  "
    f"{beamstop_two_theta_deg:.2f}° "
    f"to "
    f"{max_full_ring_two_theta_deg:.2f}°"
)


# =====================================================================
# 19. CREATE McSTAS INSTRUMENT
# =====================================================================

instrument_text = f"""
/***********************************************************************

Simple multi-material NCrystal neutron powder diffractometer

Generated automatically from Jupyter.

IMPORTANT:

A circular absorbing beamstop is positioned immediately before
the detector to remove the direct transmitted beam.

***********************************************************************/


DEFINE INSTRUMENT Simple_NCrystal_Powder(

    lambda0 = 2.0,

    dlambda = 0.02,

    string sample_cfg = "Al_sg225.ncmat"

)


TRACE


/* ================================================================
   ORIGIN
   ================================================================ */

COMPONENT origin = Progress_bar()

    AT (0, 0, 0) ABSOLUTE


/* ================================================================
   NEUTRON SOURCE
   ================================================================ */

COMPONENT source = Source_simple(

    radius = {source_radius},

    dist = {source_to_guide},

    focus_xw = {guide_width},

    focus_yh = {guide_height},

    lambda0 = lambda0,

    dlambda = dlambda,

    gauss = 0

)

    AT (0, 0, 0) RELATIVE origin


/* ================================================================
   STRAIGHT GUIDE
   ================================================================ */

COMPONENT guide = Guide(

    w1 = {guide_width},

    h1 = {guide_height},

    w2 = {guide_width},

    h2 = {guide_height},

    l = {guide_length},

    R0 = 0.99,

    Qc = 0.0219,

    alpha = 6.07,

    m = {guide_m},

    W = 0.003

)

    AT (0, 0, {guide_z}) RELATIVE origin


/* ================================================================
   COLLIMATION SLIT 1
   ================================================================ */

COMPONENT slit1 = Slit(

    xwidth = {slit1_width},

    yheight = {slit1_height}

)

    AT (0, 0, {slit1_z}) RELATIVE origin


/* ================================================================
   COLLIMATION SLIT 2
   ================================================================ */

COMPONENT slit2 = Slit(

    xwidth = {slit2_width},

    yheight = {slit2_height}

)

    AT (0, 0, {slit2_z}) RELATIVE origin


/* ================================================================
   NCRYSTAL POWDER SAMPLE
   ================================================================ */

SPLIT {sample_split} COMPONENT sample = NCrystal_sample(

    cfg = sample_cfg,

    radius = {sample_radius},

    yheight = {sample_height},

    absorptionmode = 1,

    multscat = 1

)

    AT (0, 0, {sample_z}) RELATIVE origin


/* ================================================================
   PHYSICAL DIRECT-BEAM STOP
   ================================================================

   Circular, infinitely absorbing McStas beamstop.

   Placed close to detector so the direct beam is removed
   without unnecessarily suppressing a large low-angle
   diffraction cone.

   ================================================================ */

COMPONENT direct_beam_stop = Beamstop(

    radius = {beamstop_radius}

)

    AT (0, 0, {beamstop_z}) RELATIVE origin


/* ================================================================
   2D POSITION-SENSITIVE DETECTOR
   ================================================================ */

COMPONENT detector = PSD_monitor(

    xwidth = {detector_width},

    yheight = {detector_height},

    nx = {detector_nx},

    ny = {detector_ny},

    filename = "{monitor_filename}"

)

    AT (0, 0, {detector_z}) RELATIVE origin


END
"""


instrument_file.write_text(
    instrument_text
)


# ---------------------------------------------------------------------
# Save exact instrument used with dataset
# ---------------------------------------------------------------------

instrument_snapshot = (

    sweep_root
    /
    "instrument_snapshot.instr"
)


instrument_snapshot.write_text(
    instrument_text
)


print()

print(
    "Instrument written to:"
)

print(
    instrument_file.resolve()
)


# =====================================================================
# 20. SAVE MATERIAL TABLE
# =====================================================================

materials_df = pd.DataFrame(

    materials,

    columns=[
        "material",
        "ncrystal_file"
    ]
)


materials_df.to_csv(

    sweep_root
    /
    "materials.csv",

    index=False
)


# =====================================================================
# 21. SAVE SIMULATION SETTINGS
# =====================================================================


settings = {
    "configuration_id":
        configuration_id,

    "number_of_materials":
        number_of_materials,

    "lambda1_A":
        lambda1,

    "lambda2_A":
        lambda2,

    "lambda_step_A":
        lambda_step,

    "dlambda1_A":
        dlambda1,

    "dlambda2_A":
        dlambda2,

    "dlambda_step_A":
        dlambda_step,

    "ncount":
        ncount,

    "sample_split":
        sample_split,

    "guide_length_m":
        guide_length,

    "guide_width_m":
        guide_width,

    "guide_height_m":
        guide_height,

    "guide_m":
        guide_m,

    "sample_radius_m":
        sample_radius,

    "sample_height_m":
        sample_height,

    "sample_detector_distance_m":
        sample_detector_distance,

    "detector_width_m":
        detector_width,

    "detector_height_m":
        detector_height,

    "detector_nx":
        detector_nx,

    "detector_ny":
        detector_ny,

    "beamstop_radius_m":
        beamstop_radius,

    "beamstop_detector_gap_m":
        beamstop_detector_gap,

    "beamstop_two_theta_cutoff_deg":
        beamstop_two_theta_deg,

    "profile_bins":
        profile_bins
}


with open(

    sweep_root
    /
    "settings.json",

    "w"

) as f:

    json.dump(

        settings,

        f,

        indent=4
    )


# =====================================================================
# 22. READ McSTAS 2D INTENSITY
# =====================================================================

def read_mcstas_2d_intensity(
    filename
):

    filename = Path(
        filename
    )


    lines = (

        filename
        .read_text()
        .splitlines()
    )


    data_start = None

    data_end = None


    for i, line in enumerate(
        lines
    ):

        if line.startswith(
            "# Data"
        ):

            data_start = i + 1

            break


    if data_start is None:

        raise ValueError(

            f"Could not find '# Data' in:\n"
            f"{filename}"
        )


    for i in range(
        data_start,
        len(lines)
    ):

        if lines[i].startswith(
            "#"
        ):

            data_end = i

            break


    if data_end is None:

        data_end = len(
            lines
        )


    intensity = np.array(

        [

            [
                float(value)

                for value

                in line.split()
            ]

            for line

            in lines[
                data_start:
                data_end
            ]

            if line.strip()

        ],

        dtype=np.float32
    )


    return intensity


# =====================================================================
# 23. ROBUST DISPLAY SCALING
# =====================================================================
#
# THIS IS ONLY FOR PNG PREVIEWS.
#
# It DOES NOT modify ML data.
#
# This solves the common problem where diffraction rings exist
# but appear almost completely black because of dynamic range.
#
# =====================================================================

def make_log_preview(
    image
):

    data = np.clip(

        image.astype(
            np.float64
        ),

        0,

        None
    )


    positive = data[
        data > 0
    ]


    if positive.size == 0:

        return np.zeros_like(
            data
        )


    low = np.percentile(
        positive,
        1.0
    )


    high = np.percentile(
        positive,
        99.8
    )


    if high <= low:

        high = positive.max()

        low = positive.min()


    if high <= 0:

        return np.zeros_like(
            data
        )


    low = max(
        low,
        high * 1e-6
    )


    clipped = np.clip(

        data,

        low,

        high
    )


    display_data = np.log10(
        clipped
    )


    log_low = np.log10(
        low
    )

    log_high = np.log10(
        high
    )


    if log_high > log_low:

        display_data = (

            display_data
            -
            log_low

        ) / (

            log_high
            -
            log_low
        )


    else:

        display_data = np.zeros_like(
            data
        )


    display_data[
        data <= 0
    ] = 0


    return np.clip(

        display_data,

        0,

        1
    )


# =====================================================================
# 24. RADIAL INTEGRATION
# =====================================================================
#
# Convert 2D Debye-Scherrer rings into a conventional-like
# one-dimensional powder diffractogram:
#
#                    I(2theta)
#
#
# We save BOTH:
#
# intensity_sum
#
#       integrated counts around each ring
#
# and
#
# intensity_mean
#
#       mean pixel intensity around each ring
#
#
# For ML, intensity_mean is often convenient because it removes the
# trivial increase in number of pixels with ring circumference.
#
# =====================================================================

def calculate_radial_diffractogram(
    image
):

    ny, nx = image.shape


    # -----------------------------------------------------------------
    # Pixel size in metres
    # -----------------------------------------------------------------

    dx = (
        detector_width
        /
        nx
    )

    dy = (
        detector_height
        /
        ny
    )


    # -----------------------------------------------------------------
    # Coordinates of pixel centres
    # -----------------------------------------------------------------

    x = (

        np.arange(nx)
        -
        (nx - 1) / 2
    ) * dx


    y = (

        np.arange(ny)
        -
        (ny - 1) / 2
    ) * dy


    X, Y = np.meshgrid(
        x,
        y
    )


    radius = np.sqrt(

        X**2
        +
        Y**2
    )


    # -----------------------------------------------------------------
    # Scattering angle
    #
    # Pixel radius / sample-detector distance gives the outgoing
    # ray angle relative to the incident beam.
    #
    # This is the conventional 2theta scattering angle.
    # -----------------------------------------------------------------

    two_theta = np.degrees(

        np.arctan2(

            radius,

            sample_detector_distance
        )
    )


    # -----------------------------------------------------------------
    # Only use complete circular rings.
    #
    # Otherwise high-angle bins would contain only corners/arcs
    # of the detector and produce a geometric bias.
    # -----------------------------------------------------------------

    valid = (

        radius
        <=
        full_ring_radius
    )


    # -----------------------------------------------------------------
    # Remove beamstop-shadow region from radial profile.
    #
    # The physical McStas beamstop has ALREADY removed these neutrons.
    #
    # This analysis mask prevents those zero pixels from contaminating
    # the first radial bins.
    # -----------------------------------------------------------------

    valid &= (

        two_theta
        >=
        beamstop_two_theta_deg
    )


    valid &= np.isfinite(
        image
    )


    valid &= (
        image >= 0
    )


    # -----------------------------------------------------------------
    # Angular bins
    # -----------------------------------------------------------------

    edges = np.linspace(

        beamstop_two_theta_deg,

        max_full_ring_two_theta_deg,

        profile_bins + 1
    )


    centres = (

        edges[:-1]
        +
        edges[1:]

    ) / 2


    theta_flat = two_theta[
        valid
    ]


    intensity_flat = image[
        valid
    ].astype(
        np.float64
    )


    # -----------------------------------------------------------------
    # Assign each pixel to angular bin
    # -----------------------------------------------------------------

    bin_index = np.digitize(

        theta_flat,

        edges

    ) - 1


    good = (

        (bin_index >= 0)
        &
        (bin_index < profile_bins)
    )


    bin_index = bin_index[
        good
    ]


    intensity_flat = intensity_flat[
        good
    ]


    # -----------------------------------------------------------------
    # SUM intensity
    # -----------------------------------------------------------------

    intensity_sum = np.bincount(

        bin_index,

        weights=intensity_flat,

        minlength=profile_bins
    ).astype(
        np.float64
    )


    # -----------------------------------------------------------------
    # Pixel count
    # -----------------------------------------------------------------

    pixel_count = np.bincount(

        bin_index,

        minlength=profile_bins
    ).astype(
        np.int64
    )


    # -----------------------------------------------------------------
    # MEAN intensity
    # -----------------------------------------------------------------

    intensity_mean = np.divide(

        intensity_sum,

        pixel_count,

        out=np.zeros_like(
            intensity_sum
        ),

        where=pixel_count > 0
    )


    # -----------------------------------------------------------------
    # Normalized copy for convenience.
    #
    # ORIGINAL intensity_mean is also retained!
    # -----------------------------------------------------------------

    maximum = intensity_mean.max()


    if maximum > 0:

        intensity_mean_unitmax = (

            intensity_mean
            /
            maximum
        )

    else:

        intensity_mean_unitmax = np.zeros_like(
            intensity_mean
        )


    return (

        centres,

        intensity_sum,

        intensity_mean,

        intensity_mean_unitmax,

        pixel_count
    )


# =====================================================================
# 25. DATASET SIZE
# =====================================================================

n_materials = len(
    materials
)

n_lambda = len(
    lambdas
)

n_dlambda = len(
    dlambdas
)


total_simulations = (

    n_materials
    *
    n_lambda
    *
    n_dlambda
)


print()

print("=" * 80)
print("DATASET SIZE")
print("=" * 80)

print()

print(
    f"Materials         = {n_materials}"
)

print(
    f"Lambda values     = {n_lambda}"
)

print(
    f"dlambda values    = {n_dlambda}"
)

print()

print(
    f"TOTAL             = "
    f"{n_materials} × "
    f"{n_lambda} × "
    f"{n_dlambda} "
    f"= {total_simulations} simulations"
)

print()


# =====================================================================
# 26. RUN EVERYTHING
# =====================================================================

master_metadata = []

simulation_index = 0


for material_label, ncmat_file in materials:


    # =================================================================
    # MATERIAL FOLDERS
    # =================================================================

    material_dir = (

        sweep_root
        /
        material_label
    )


    patterns_2d_dir = (

        material_dir
        /
        "patterns_2D"
    )


    previews_2d_dir = (

        material_dir
        /
        "previews_2D"
    )


    profiles_1d_dir = (

        material_dir
        /
        "profiles_1D"
    )


    previews_1d_dir = (

        material_dir
        /
        "previews_1D"
    )


    runs_dir = (

        material_dir
        /
        "mcstas_runs"
    )


    for directory in [

        patterns_2d_dir,

        previews_2d_dir,

        profiles_1d_dir,

        previews_1d_dir,

        runs_dir

    ]:

        directory.mkdir(

            parents=True,

            exist_ok=True
        )


    material_metadata = []


    print()

    print("=" * 80)

    print(
        f"MATERIAL: {material_label}"
    )

    print(
        f"NCrystal: {ncmat_file}"
    )

    print("=" * 80)


    # =================================================================
    # PARAMETER SWEEP
    # =================================================================

    for wavelength in lambdas:

        for dlambda in dlambdas:


            simulation_index += 1


            # ---------------------------------------------------------
            # NAME
            # ---------------------------------------------------------

            lambda_tag = number_to_tag(
                wavelength
            )


            dlambda_tag = number_to_tag(
                dlambda
            )


            sample_name = (

                f"{material_label}"

                f"_lambda_"
                f"{lambda_tag}"

                f"_dlambda_"
                f"{dlambda_tag}"
            )


            # ---------------------------------------------------------
            # FILE LOCATIONS
            # ---------------------------------------------------------

            run_dir = (

                runs_dir
                /
                sample_name
            )


            pattern_2d_file = (

                patterns_2d_dir
                /
                f"{sample_name}.npy"
            )


            preview_2d_file = (

                previews_2d_dir
                /
                f"{sample_name}.png"
            )


            profile_npz_file = (

                profiles_1d_dir
                /
                f"{sample_name}_profile.npz"
            )


            profile_csv_file = (

                profiles_1d_dir
                /
                f"{sample_name}_profile.csv"
            )


            profile_preview_file = (

                previews_1d_dir
                /
                f"{sample_name}_profile.png"
            )


            print()

            print(

                f"["
                f"{simulation_index:05d}"
                f"/"
                f"{total_simulations:05d}"
                f"] "

                f"{material_label:<15} | "

                f"λ = "
                f"{wavelength:.4f} Å | "

                f"dλ = "
                f"{dlambda:.4f} Å"
            )


            # =========================================================
            # EXISTING PATTERN?
            # =========================================================

            if (

                pattern_2d_file.exists()

                and

                not overwrite_existing

            ):

                image = np.load(
                    pattern_2d_file
                )

                status = "existing"


            else:


                # -----------------------------------------------------
                # Clean old run
                # -----------------------------------------------------

                if run_dir.exists():

                    shutil.rmtree(
                        run_dir
                    )


                # -----------------------------------------------------
                # McStas command
                # -----------------------------------------------------

                command = [

                    mcrun_executable,

                    str(
                        instrument_file
                    ),

                    f"lambda0="
                    f"{wavelength:.10f}",

                    f"dlambda="
                    f"{dlambda:.10f}",

                    f"sample_cfg="
                    f"{ncmat_file}",

                    "-n",

                    str(
                        ncount
                    ),

                    "-d",

                    str(
                        run_dir
                    )
                ]


                # -----------------------------------------------------
                # RUN
                # -----------------------------------------------------

                result = subprocess.run(

                    command,

                    capture_output=True,

                    text=True
                )


                # -----------------------------------------------------
                # Error handling
                # -----------------------------------------------------

                if result.returncode != 0:

                    print()

                    print(
                        "❌ McStas FAILED"
                    )

                    print()

                    print(
                        "--- STDOUT ---"
                    )

                    print(
                        result.stdout
                    )

                    print()

                    print(
                        "--- STDERR ---"
                    )

                    print(
                        result.stderr
                    )


                    raise RuntimeError(

                        "\nSimulation failed:\n"

                        f"material = {material_label}\n"

                        f"NCrystal = {ncmat_file}\n"

                        f"lambda   = {wavelength}\n"

                        f"dlambda  = {dlambda}\n"
                    )


                # -----------------------------------------------------
                # Detector file
                # -----------------------------------------------------

                detector_file = (

                    run_dir
                    /
                    monitor_filename
                )


                if not detector_file.exists():

                    raise FileNotFoundError(

                        "\nMcStas completed but rings.dat "
                        "was not created:\n\n"

                        f"{detector_file}"
                    )


                # -----------------------------------------------------
                # Read diffraction image
                # -----------------------------------------------------

                image = read_mcstas_2d_intensity(

                    detector_file
                )


                # -----------------------------------------------------
                # Save ORIGINAL 2D intensities
                # -----------------------------------------------------

                np.save(

                    pattern_2d_file,

                    image.astype(
                        np.float32
                    )
                )


                status = "simulated"


            # =================================================================
            # 27. DYNAMIC-RANGE CORRECTED 2D PREVIEW
            # =================================================================

            if save_2d_previews:

                display_image = make_log_preview(
                    image
                )


                plt.imsave(

                    preview_2d_file,

                    display_image,

                    cmap="inferno",

                    origin="lower",

                    vmin=0,

                    vmax=1
                )


            # =================================================================
            # 28. CALCULATE 1D POWDER DIFFRACTOGRAM
            # =================================================================

            (

                two_theta_deg,

                intensity_sum,

                intensity_mean,

                intensity_mean_unitmax,

                pixel_count

            ) = calculate_radial_diffractogram(
                image
            )


            # =================================================================
            # 29. SAVE 1D PROFILE AS NPZ
            # =================================================================
            #
            # Best format for ML.
            #
            # =================================================================

            np.savez_compressed(

                profile_npz_file,

                two_theta_deg=
                    two_theta_deg.astype(
                        np.float32
                    ),

                intensity_sum=
                    intensity_sum.astype(
                        np.float32
                    ),

                intensity_mean=
                    intensity_mean.astype(
                        np.float32
                    ),

                intensity_mean_unitmax=
                    intensity_mean_unitmax.astype(
                        np.float32
                    ),

                pixel_count=
                    pixel_count.astype(
                        np.int32
                    )
            )


            # =================================================================
            # 30. SAVE 1D PROFILE AS CSV
            # =================================================================
            #
            # Easy for humans to inspect.
            #
            # =================================================================

            profile_df = pd.DataFrame(

                {

                    "two_theta_deg":
                        two_theta_deg,

                    "intensity_sum":
                        intensity_sum,

                    "intensity_mean":
                        intensity_mean,

                    "intensity_mean_unitmax":
                        intensity_mean_unitmax,

                    "pixel_count":
                        pixel_count
                }
            )


            profile_df.to_csv(

                profile_csv_file,

                index=False
            )


            # =================================================================
            # 31. SAVE 1D DIFFRACTOGRAM PREVIEW
            # =================================================================

            if save_1d_previews:

                fig = plt.figure(

                    figsize=(
                        8,
                        4.5
                    )
                )


                plt.plot(

                    two_theta_deg,

                    intensity_mean
                )


                plt.xlabel(
                    r"$2\theta$ [deg]"
                )


                plt.ylabel(
                    "Mean detector intensity"
                )


                plt.title(

                    f"{material_label} | "
                    rf"$\lambda={wavelength:.3f}$ Å | "
                    rf"$\Delta\lambda={dlambda:.3f}$ Å"
                )


                plt.yscale(
                    "symlog",
                    linthresh=1e-12
                )


                plt.grid(
                    alpha=0.25
                )


                plt.tight_layout()


                plt.savefig(

                    profile_preview_file,

                    dpi=150
                )


                plt.close(
                    fig
                )


            # =================================================================
            # 32. QUALITY DIAGNOSTICS
            # =================================================================

            positive_pixels = np.count_nonzero(
                image > 0
            )


            nonzero_fraction = (

                positive_pixels
                /
                image.size
            )


            total_intensity = float(
                image.sum()
            )


            max_intensity = float(
                image.max()
            )


            # =================================================================
            # 33. METADATA
            # =================================================================

            record = {

                "sample_name":
                    sample_name,

                "material":
                    material_label,

                "ncrystal_file":
                    ncmat_file,

                "lambda_A":
                    float(
                        wavelength
                    ),

                "dlambda_A":
                    float(
                        dlambda
                    ),

                "ncount":
                    int(
                        ncount
                    ),

                "beamstop_radius_mm":
                    beamstop_radius
                    *
                    1000,

                "beamstop_2theta_cutoff_deg":
                    beamstop_two_theta_deg,

                "shape_y":
                    int(
                        image.shape[0]
                    ),

                "shape_x":
                    int(
                        image.shape[1]
                    ),

                "max_intensity":
                    max_intensity,

                "sum_intensity":
                    total_intensity,

                "nonzero_fraction":
                    float(
                        nonzero_fraction
                    ),

                "pattern_2d_file":
                    str(

                        pattern_2d_file.relative_to(
                            sweep_root
                        )
                    ),

                "profile_1d_npz":
                    str(

                        profile_npz_file.relative_to(
                            sweep_root
                        )
                    ),

                "profile_1d_csv":
                    str(

                        profile_csv_file.relative_to(
                            sweep_root
                        )
                    ),

                "status":
                    status
            }


            master_metadata.append(
                record
            )


            material_metadata.append(
                record
            )


            # =================================================================
            # 34. PRINT DIAGNOSTIC
            # =================================================================

            print(

                f"    ✓ {status:<9} | "

                f"max={max_intensity:.3e} | "

                f"sum={total_intensity:.3e} | "

                f"nonzero="
                f"{100*nonzero_fraction:.3f}%"
            )


            if total_intensity <= 0:

                print(

                    "    ⚠ WARNING: detector contains "
                    "zero total intensity."
                )


    # =================================================================
    # 35. SAVE METADATA FOR EACH MATERIAL
    # =================================================================

    material_metadata_df = pd.DataFrame(

        material_metadata
    )


    material_metadata_df.to_csv(

        material_dir
        /
        "metadata.csv",

        index=False
    )


# =====================================================================
# 36. MASTER METADATA
# =====================================================================

master_metadata_df = pd.DataFrame(

    master_metadata
)


master_metadata_file = (

    sweep_root
    /
    "master_metadata.csv"
)


master_metadata_df.to_csv(

    master_metadata_file,

    index=False
)


# =====================================================================
# 37. FINAL SUMMARY
# =====================================================================

print()

print("=" * 80)

print("DATASET COMPLETE")

print("=" * 80)

print()

print(
    f"Materials:        "
    f"{n_materials}"
)

print(
    f"2D patterns:      "
    f"{len(master_metadata_df)}"
)

print(
    f"1D profiles:      "
    f"{len(master_metadata_df)}"
)

print()

print(
    "Dataset location:"
)

print(
    sweep_root.resolve()
)

print()

print(
    "Master metadata:"
)

print(
    master_metadata_file.resolve()
)

print()

print(
    "Instrument:"
)

print(
    instrument_snapshot.resolve()
)

print()

print("=" * 80)


# =====================================================================
# 38. SHOW FIRST ENTRIES
# =====================================================================

display(

    master_metadata_df.head(
        20
    )
)


# =====================================================================
# 39. SHOW ONE EXAMPLE RESULT
# =====================================================================
#
# Display:
#
#       left  = robustly scaled 2D diffraction pattern
#       right = radial 1D diffractogram
#
# =====================================================================

if len(master_metadata_df) > 0:

    example = master_metadata_df.iloc[
        0
    ]


    example_image = np.load(

        sweep_root
        /
        example[
            "pattern_2d_file"
        ]
    )


    example_profile = np.load(

        sweep_root
        /
        example[
            "profile_1d_npz"
        ]
    )


    fig, ax = plt.subplots(

        1,
        2,

        figsize=(
            13,
            5
        )
    )


    # -----------------------------------------------------------------
    # 2D
    # -----------------------------------------------------------------

    ax[0].imshow(

        make_log_preview(
            example_image
        ),

        origin="lower",

        cmap="inferno",

        vmin=0,

        vmax=1
    )


    ax[0].set_title(

        f"{example['material']}\n"

        rf"$\lambda="
        f"{example['lambda_A']:.3f}$ Å, "

        rf"$\Delta\lambda="
        f"{example['dlambda_A']:.3f}$ Å"
    )


    ax[0].set_xlabel(
        "Detector x [pixel]"
    )


    ax[0].set_ylabel(
        "Detector y [pixel]"
    )


    # -----------------------------------------------------------------
    # 1D
    # -----------------------------------------------------------------

    ax[1].plot(

        example_profile[
            "two_theta_deg"
        ],

        example_profile[
            "intensity_mean"
        ]
    )


    ax[1].set_xlabel(

        r"$2\theta$ [deg]"
    )


    ax[1].set_ylabel(

        "Mean radial intensity"
    )


    ax[1].set_title(

        "Azimuthally integrated "
        "powder diffractogram"
    )


    ax[1].set_yscale(

        "symlog",

        linthresh=1e-12
    )


    ax[1].grid(

        alpha=0.25
    )


    plt.tight_layout()

    plt.show()